# __DATA 22700 Final Project: Jupyter Notebook__
## Following deportation trends in the US for the last ten years (2015 - 2025)
### Bernardo Frank, Mateo Porter, Kevin Solano

# *Introduction*

This project is intended to follow trends in deportation data from two sources: the U.S. Immigration and Customs Enforcement statistics site and the Deportation Data site on historical deportation data.

### How has ICE detention changed over time?
#### What this can reveal: variables that are tied into certain changes in ICE detention and deportation rates

narrative hook: ...

# *Data provenance and DataFrame cleaning*

In [4]:
import pandas as pd
import altair as alt
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

In [13]:
ice_arrests = pd.read_excel("./data/ICE_data.xlsx")
ice_arrests.info()

<class 'pandas.DataFrame'>
RangeIndex: 9707 entries, 0 to 9706
Data columns (total 8 columns):
 #   Column                        Non-Null Count  Dtype
---  ------                        --------------  -----
 0   Criminality                   9707 non-null   str  
 1   Area of Responsibility (AOR)  9707 non-null   str  
 2   Country of Citizenship        9707 non-null   str  
 3   Fiscal Year                   9707 non-null   int64
 4   Fiscal Quarter                9707 non-null   int64
 5   Fiscal Month                  9707 non-null   int64
 6   Month-Year                    9707 non-null   str  
 7   Administrative Arrests        9707 non-null   int64
dtypes: int64(4), str(4)
memory usage: 606.8 KB


Provenance: U.S. Immigrations and Customs Enforcement
Link: https://www.ice.gov/statistics

Within the ICE Excel data sheet, there are 5 relevant tables that we used in our exploration

In [11]:
ice_atd = pd.read_excel("./streamlit_fodler/data/ICE_data.xlsx", sheet_name="ICE ATD")
ice_arrests = pd.read_excel("./streamlit_fodler/data/ICE_data.xlsx", sheet_name="ICE-ERO Administrative Arrests")
ice_detentions = pd.read_excel("./streamlit_fodler/data/ICE_data.xlsx", sheet_name="ICE Detentions")
ice_removals = pd.read_excel("./streamlit_fodler/data/ICE_data.xlsx", sheet_name="ICE Removals")
ice_ex_individuals = pd.read_excel("./streamlit_fodler/data/ICE_data.xlsx", sheet_name='ICE T42 Expulsions Indivduals')
ice_ex_flights = pd.read_excel("./streamlit_fodler/data/ICE_data.xlsx", sheet_name='ICE T42 Expulsions Flights ')

## ICE Data
The ICE Data contains five relevant tables:

ICE Alternatives to Detention Data: Featuring numbers of migrants who are not physically detained but are monitored using other methods.
ICE Arrests Data: Featuring the number of individuals arrested by ICE. Different from detentions, arrests mean an officer apprehended someone.
ICE Detentions Data: Featuring the number of individuals detained by ICE. Different from arrests, detentions mean an individual was held in custody at a detention center.
ICE Removals Data: Featuring numbers of individuals that were deported from the U.S.
ICE T42 Expulsions Individual Data: Features the number of expulsion of individuals under the Title 42 (T42) order, which allows the expulsion of individuals from the country without normal immigration processing.
ICE T42 Expulsions Flight Data: Features the number of T42 Expulsion related flights.

Now, to visualize yearly ICE activity, we can combine the tables of ICE data into one 

### __Data Table Example #1__

In [ ]:
# ICE Alternatives to Detention Data

filtered_atd_df = ice_atd[ice_atd[column_1] == value_1]

filtered_atd_df


## __Visualizations__

### ICE Enforcement Actions by Fiscal Year

To visualize this, we compiled the tables from ICE data sheets into one DataFrame, where each row is a Fiscal Year and columns represents counts of arrests, detentions, removals, T42 expulsions, and ATD.

Altair modules generate the visualization where each bar and color represents a Fiscal Year. The Y axis represents counts for each bar, users can select which form of enforcement to track.

In [16]:
atd_yearly = ice_atd.groupby("Fiscal Year").size().reset_index(name="ATD")
arrests_yearly = ice_arrests.groupby("Fiscal Year").size().reset_index(name="Arrests")
detentions_yearly = ice_detentions.groupby("Fiscal Year").size().reset_index(name="Detentions")
removals_yearly = ice_removals.groupby("Fiscal Year").size().reset_index(name="Removals")
expulsions_yearly = ice_ex_individuals.groupby("Fiscal Year").size().reset_index(name="T42 Expulsions")

combined = atd_yearly.merge(arrests_yearly, on="Fiscal Year", how="outer") \
    .merge(detentions_yearly, on="Fiscal Year", how="outer") \
    .merge(removals_yearly, on="Fiscal Year", how="outer") \
    .merge(expulsions_yearly, on="Fiscal Year", how="outer")
combined = combined.fillna(0)

combined["Total"] = (
    combined["ATD"] +
    combined["Arrests"] +
    combined["Detentions"] +
    combined["Removals"] +
    combined["T42 Expulsions"]
)

long_df = combined.melt(
    id_vars="Fiscal Year",
    var_name="Enforcement_Type",
    value_name="Count"
)

selector = alt.selection_point(
    fields=["Enforcement_Type"],
    bind=alt.binding_select(
        options=[
            "ATD",
            "Arrests",
            "Detentions",
            "Removals",
            "T42 Expulsions",
            "Total"
        ],
        name="Enforcement Type: "
    ),
)

In [17]:
chart = (
    alt.Chart(long_df)
    .mark_bar()
    .encode(
        x=alt.X("Fiscal Year:O", title="Fiscal Year"),
        y=alt.Y("Count:Q", title="Count"),
        color="Fiscal Year:O",
        tooltip=["Fiscal Year", "Enforcement_Type", "Count"]
    )
    .add_params(selector)
    .transform_filter(selector)
    .properties(
        title="ICE Enforcement Actions by Fiscal Year"
    )
)
chart

alt.Chart(...)

Another important aspect to notice is that to fully track this statistic we would need individual data that can be traced across different tables. This is extremely sensitive data and should only be handled by responsible authorities. To account for this, we decided calculate ratios of for example the number of detentions from 2024-2025 and divide by the number of arrests from the first year. This will give a percentage estimate of arrests in the two years that led to detentions. The choice between two years is due to the fact that ceratin individuals may have been arrested or detained in one year but only processed in the next. The following graph shows a logical understanding of what the pipeline looks like.

In [22]:
combined = combined.sort_values("Fiscal Year")

combined["Detentions_next"] = combined["Detentions"].shift(-1)
combined["Removals_next"] = combined["Removals"].shift(-1)
combined["ATD_next"] = combined["ATD"].shift(-1)

combined["Arrest_to_Detention"] = (
    (combined["Detentions"] + combined["Detentions_next"]) /
    combined["Arrests"]
)

combined["Arrest_to_ATD"] = (
    (combined["ATD"] + combined["ATD_next"]) /
    combined["Arrests"]
)

combined["Detention_to_Removal"] = (
    (combined["Removals"] + combined["Removals_next"]) /
    combined["Detentions"]
)

combined["Detention_to_ATD"] = (
    (combined["ATD"] + combined["ATD_next"]) /
    combined["Detentions"]
)

pipeline = combined.iloc[1:-1]

pipeline_melt = pipeline.melt(
    id_vars="Fiscal Year",
    value_vars=["Arrest_to_Detention", "Detention_to_Removal", "Arrest_to_ATD", "Detention_to_ATD"],
    var_name="Stage",
    value_name="Rate"
)

atd_chart = alt.Chart(pipeline_melt).mark_line(point=True).encode(
    x=alt.X(
        "Fiscal Year:O",
        axis=alt.Axis(
            title="Fiscal Year Window",
            labelExpr="datum.label + '–' + (parseInt(datum.label) + 1)",
            labelAngle=0
        )
    ),
    y="Rate:Q",
    color="Stage:N",
    tooltip=["Fiscal Year", "Stage", "Rate"]
).properties(width= 700, height= 500)

atd_chart


alt.Chart(...)

Note how the edges leading towards ATD are usually higher than the ones leading towards either detention or removal from the same source. This indicates that law enforcement and legislators tend to prefer alternative methods of enforcement rather than following the traditional pipeline of arrest → detention → removal.

Note also that there was a slight decrease towards ATD punishements and increase in detentions or removals between 2022-2023 and 2023-2024. We couldn't find any news relating to this change, but this could simply be due to more effective methods of detection which increased the number of arrests and detentions.

### State-level arrests by gender

In [ ]:
# loading new data sets for 2025 to 2026 analysis
ICE_arrest_25 = pd.read_csv("./streamlit_folder/data/ICE25.csv", encoding= "latin-1")
ICE_arrest_26 = pd.read_csv("./streamlit_folder/data/ICE26(sheet1).csv", encoding= "latin-1")

In [ ]:
# cleaning relevant columns
def clean_numeric(series):
    return (
        series.astype(str)
        .str.replace(",", "", regex=False)
        .str.strip()
        .replace({"nan": None, "": None})
        .astype(float)
        .fillna(0)
        .astype(int)
    )

for df in [ICE_arrest_25, ICE_arrest_26]:
    df["Male Non-Crim"] = clean_numeric(df["Male Non-Crim"])
    df["Male Crim"] = clean_numeric(df["Male Crim"])
    df["Female Non-Crim"] = clean_numeric(df["Female Non-Crim"])
    df["Female Crim"] = clean_numeric(df["Female Crim"])

    df["Men"] = df["Male Crim"] + df["Male Non-Crim"]
    df["Women"] = df["Female Crim"] + df["Female Non-Crim"]

state_totals25 = ICE_arrest_25.groupby("State")[["Men", "Women"]].sum().reset_index()
state_totals26 = ICE_arrest_26.groupby("State")[["Men", "Women"]].sum().reset_index()

In [ ]:
# formatting for the json file
state_debrev = {
    "AL": "Alabama", "AK": "Alaska", "AZ": "Arizona", "AR": "Arkansas",
    "CA": "California", "CO": "Colorado", "CT": "Connecticut", "DE": "Delaware",
    "FL": "Florida", "GA": "Georgia", "HI": "Hawaii", "ID": "Idaho",
    "IL": "Illinois", "IN": "Indiana", "IA": "Iowa", "KS": "Kansas",
    "KY": "Kentucky", "LA": "Louisiana", "ME": "Maine", "MD": "Maryland",
    "MA": "Massachusetts", "MI": "Michigan", "MN": "Minnesota", "MS": "Mississippi",
    "MO": "Missouri", "MT": "Montana", "NE": "Nebraska", "NV": "Nevada",
    "NH": "New Hampshire", "NJ": "New Jersey", "NM": "New Mexico", "NY": "New York",
    "NC": "North Carolina", "ND": "North Dakota", "OH": "Ohio", "OK": "Oklahoma",
    "OR": "Oregon", "PA": "Pennsylvania", "RI": "Rhode Island", "SC": "South Carolina",
    "SD": "South Dakota", "TN": "Tennessee", "TX": "Texas", "UT": "Utah",
    "VT": "Vermont", "VA": "Virginia", "WA": "Washington", "WV": "West Virginia",
    "WI": "Wisconsin", "WY": "Wyoming"
}

state_totals25["State Name"] = state_totals25["State"].map(state_debrev)
state_totals26["State Name"] = state_totals26["State"].map(state_debrev)

# FIPS ids for state map
state_fips = {
    "Alabama": 1, "Alaska": 2, "Arizona": 4, "Arkansas": 5, "California": 6,
    "Colorado": 8, "Connecticut": 9, "Delaware": 10, "District of Columbia": 11,
    "Florida": 12, "Georgia": 13, "Hawaii": 15, "Idaho": 16, "Illinois": 17,
    "Indiana": 18, "Iowa": 19, "Kansas": 20, "Kentucky": 21, "Louisiana": 22,
    "Maine": 23, "Maryland": 24, "Massachusetts": 25, "Michigan": 26,
    "Minnesota": 27, "Mississippi": 28, "Missouri": 29, "Montana": 30,
    "Nebraska": 31, "Nevada": 32, "New Hampshire": 33, "New Jersey": 34,
    "New Mexico": 35, "New York": 36, "North Carolina": 37, "North Dakota": 38,
    "Ohio": 39, "Oklahoma": 40, "Oregon": 41, "Pennsylvania": 42,
    "Rhode Island": 44, "South Carolina": 45, "South Dakota": 46,
    "Tennessee": 47, "Texas": 48, "Utah": 49, "Vermont": 50, "Virginia": 51,
    "Washington": 53, "West Virginia": 54, "Wisconsin": 55, "Wyoming": 56
}

# state names in the dataframe by the same key (id) used by the map
state_totals25["id"] = state_totals25["State Name"].map(state_fips)
state_totals26["id"] = state_totals26["State Name"].map(state_fips)

# cleaning to avoid missing data
state_totals25 = state_totals25.dropna(subset=["id"]).copy()
state_totals25["id"] = state_totals25["id"].astype(int)

state_totals26 = state_totals26.dropna(subset=["id"]).copy()
state_totals26["id"] = state_totals26["id"].astype(int)

# state chloropleths

#this is the main change ---> Bypasses PyArrow Serialization
us_map = alt.topo_feature("https://vega.github.io/vega-datasets/data/us-10m.json",feature="states")

selection = alt.selection_point(fields=['State Name'], empty = 'none')

state_debrev = {
    "AL": "Alabama", "AK": "Alaska", "AZ": "Arizona", "AR": "Arkansas",
    "CA": "California", "CO": "Colorado", "CT": "Connecticut", "DE": "Delaware",
    "FL": "Florida", "GA": "Georgia", "HI": "Hawaii", "ID": "Idaho",
    "IL": "Illinois", "IN": "Indiana", "IA": "Iowa", "KS": "Kansas",
    "KY": "Kentucky", "LA": "Louisiana", "ME": "Maine", "MD": "Maryland",
    "MA": "Massachusetts", "MI": "Michigan", "MN": "Minnesota", "MS": "Mississippi",
    "MO": "Missouri", "MT": "Montana", "NE": "Nebraska", "NV": "Nevada",
    "NH": "New Hampshire", "NJ": "New Jersey", "NM": "New Mexico", "NY": "New York",
    "NC": "North Carolina", "ND": "North Dakota", "OH": "Ohio", "OK": "Oklahoma",
    "OR": "Oregon", "PA": "Pennsylvania", "RI": "Rhode Island", "SC": "South Carolina",
    "SD": "South Dakota", "TN": "Tennessee", "TX": "Texas", "UT": "Utah",
    "VT": "Vermont", "VA": "Virginia", "WA": "Washington", "WV": "West Virginia",
    "WI": "Wisconsin", "WY": "Wyoming"
}

state_totals25["State Name"] = state_totals25["State"].map(state_debrev)
state_totals26["State Name"] = state_totals26["State"].map(state_debrev)

# FIPS ids for state map
state_fips = {
    "Alabama": 1, "Alaska": 2, "Arizona": 4, "Arkansas": 5, "California": 6,
    "Colorado": 8, "Connecticut": 9, "Delaware": 10, "District of Columbia": 11,
    "Florida": 12, "Georgia": 13, "Hawaii": 15, "Idaho": 16, "Illinois": 17,
    "Indiana": 18, "Iowa": 19, "Kansas": 20, "Kentucky": 21, "Louisiana": 22,
    "Maine": 23, "Maryland": 24, "Massachusetts": 25, "Michigan": 26,
    "Minnesota": 27, "Mississippi": 28, "Missouri": 29, "Montana": 30,
    "Nebraska": 31, "Nevada": 32, "New Hampshire": 33, "New Jersey": 34,
    "New Mexico": 35, "New York": 36, "North Carolina": 37, "North Dakota": 38,
    "Ohio": 39, "Oklahoma": 40, "Oregon": 41, "Pennsylvania": 42,
    "Rhode Island": 44, "South Carolina": 45, "South Dakota": 46,
    "Tennessee": 47, "Texas": 48, "Utah": 49, "Vermont": 50, "Virginia": 51,
    "Washington": 53, "West Virginia": 54, "Wisconsin": 55, "Wyoming": 56
}

# state names in the dataframe by the same key (id) used by the map
state_totals25["id"] = state_totals25["State Name"].map(state_fips)
state_totals26["id"] = state_totals26["State Name"].map(state_fips)

# cleaning to avoid missing data
state_totals25 = state_totals25.dropna(subset=["id"]).copy()
state_totals25["id"] = state_totals25["id"].astype(int)

state_totals26 = state_totals26.dropna(subset=["id"]).copy()
state_totals26["id"] = state_totals26["id"].astype(int)

us_map = alt.topo_feature("https://vega.github.io/vega-datasets/data/us-10m.json",feature="states")
selection = alt.selection_point(fields=['State Name'], empty = 'none')


In [30]:
ICE_detention_W25 = (
    alt.Chart(us_map)
    .mark_geoshape(stroke="white", strokeWidth=0.5)
    .transform_lookup(
        lookup="id",
        #lookup by id and then use the state name
        from_=alt.LookupData(state_totals25, key="id", fields=["State Name", "Women"])
    ).encode(
        color=alt.Color("Women:Q", scale=alt.Scale(scheme="blues"), title="Women"),
        tooltip=["State Name:N", alt.Tooltip("Women:Q", format=",")],
        opacity = alt.condition(selection, alt.value(1), alt.value(0.3))
    ).add_params(
        selection
    ).project(
        type="albersUsa"
    ).properties(
        width=400, height=250, title="Women detained by ICE 2025"
    )
)

ICE_detention_M25 = (
    alt.Chart(us_map)
    .mark_geoshape(stroke="white", strokeWidth=0.5)
    .transform_lookup(
        lookup="id",
        #lookup by id and then use the state name
        from_=alt.LookupData(state_totals25, key="id", fields=["State Name", "Men"])
    ).encode(
        color=alt.Color("Men:Q", scale=alt.Scale(scheme="reds"), title="Men"),
        tooltip=["State Name:N", alt.Tooltip("Men:Q", format=",")],
        opacity = alt.condition(selection, alt.value(1), alt.value(0.3))
    ).add_params(
        selection
    ).project(
        type="albersUsa"
    ).properties(
        width=400, height=250, title="Men detained by ICE 2025"
    )
)

ICE_detention_W26 = (
    alt.Chart(us_map)
    .mark_geoshape(stroke="white", strokeWidth=0.5)
    .transform_lookup(
        lookup="id",
        #lookup by id and then use the state name
        from_=alt.LookupData(state_totals26, key="id", fields=["State Name", "Women"])
    )
    .encode(
        color=alt.Color("Women:Q", scale=alt.Scale(scheme="blues"), title="Men"),
        tooltip=["State Name:N", alt.Tooltip("Women:Q", format=",")],
        opacity = alt.condition(selection, alt.value(1), alt.value(0.3))
    ).add_params(
        selection
    )
    .project(type="albersUsa")
    .properties(width=400, height=250, title="Women detained by ICE 2026")
)

ICE_detention_M26 = (
    alt.Chart(us_map)
    .mark_geoshape(stroke="white", strokeWidth=0.5)
    .transform_lookup(
        lookup="id",
        #lookup by id and then use the state name
        from_=alt.LookupData(state_totals26, key="id", fields=["State Name", "Men"])
    )
    .encode(
        color=alt.Color("Men:Q", scale=alt.Scale(scheme="reds"), title="Men"),
        tooltip=["State Name:N", alt.Tooltip("Men:Q", format=",")],
        opacity = alt.condition(selection, alt.value(1), alt.value(0.3))
    ).add_params(
        selection
    ).project(
        type="albersUsa"
    ).properties(
        width=400, height=250, title="Men detained by ICE 2026"
    )
)

ICE_detention_25 = (ICE_detention_W25 | ICE_detention_M25).resolve_scale(color="independent")
ICE_detention_26 = (ICE_detention_W26 | ICE_detention_M26).resolve_scale(color="independent")

detention_dashboard = alt.vconcat(
    ICE_detention_26,
    ICE_detention_25
).resolve_scale(
    color='shared'
)
detention_dashboard

/var/folders/08/jrlvkhqn0gqgrd3myzkfrj_m0000gn/T/ipykernel_14219/1006665190.py:81: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  ICE_detention_25 = (ICE_detention_W25 | ICE_detention_M25).resolve_scale(color="independent")
/var/folders/08/jrlvkhqn0gqgrd3myzkfrj_m0000gn/T/ipykernel_14219/1006665190.py:82: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  ICE_detention_26 = (ICE_detention_W26 | ICE_detention_M26).resolve_scale(color="independent")
/Users/kevinsolano/micromamba/envs/venv/lib/python3.14/site-packages/IPython/core/interactiveshell.py:3701: UserWarning: Automatically deduplicated select

alt.VConcatChart(...)